In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Rozpoczęto nasłuchiwanie transakcji...")

for message in consumer:
    transaction = message.value
    
    # Sprawdzenie warunku kwoty
    if transaction.get('amount', 0) > 3000:
        print(f"ALERT: Wykryto transakcję powyżej 1000! Szczegóły: {transaction}")
    else:
        # Opcjonalnie: logowanie pominiętych transakcji
        print(f"Przetworzono transakcję: {transaction['amount']}")

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrichment_worker_v1',  # Nowy group_id pozwala na niezależne czytanie
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest'     # Opcjonalne: czytaj od początku, jeśli grupa jest nowa
)

print("Rozpoczęto wzbogacanie transakcji...")

for message in consumer:
    transaction = message.value
    amount = transaction.get('amount', 0)
    
    # Logika klasyfikacji ryzyka
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    # Wzbogacenie słownika o nowe pole
    transaction['risk_level'] = risk_level
    
    # Wypisanie wzbogaconego eventu
    print(f"Przetworzono: {transaction}")

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest',
    group_id='analytics_consumer_v1'
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Analiza strumienia rozpoczęta...")

for message in consumer:
    transaction = message.value
    store = transaction.get('store', 'Nieznany')
    amount = transaction.get('amount', 0)
    
    # 1. Zwiększ liczbę transakcji dla danego sklepu
    store_counts[store] += 1
    
    # 2. Dodaj kwotę do sumy dla danego sklepu
    total_amount[store] = total_amount.get(store, 0) + amount
    
    # Zwiększ całkowity licznik wiadomości
    msg_count += 1
    
    # 3. Co 10 wiadomości wypisz tabelę
    if msg_count % 10 == 0:
        print(f"\n--- Raport po {msg_count} wiadomościach ---")
        print(f"{'Sklep':<15} | {'Liczba':<8} | {'Suma':<10} | {'Średnia':<10}")
        print("-" * 55)
        
        for s in sorted(store_counts.keys()):
            cnt = store_counts[s]
            total = total_amount[s]
            avg = total / cnt if cnt > 0 else 0
            print(f"{s:<15} | {cnt:<8} | {total:<10.2f} | {avg:<10.2f}")

Writing consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

# Inicjalizacja struktury danych: każda nowa kategoria dostaje zestaw startowy
category_stats = defaultdict(lambda: {
    'count': 0, 
    'total': 0.0, 
    'min': float('inf'), 
    'max': float('-inf')
})

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='stats_analyzer_v1',
    auto_offset_reset='earliest'
)

msg_count = 0

print("Rozpoczęto zaawansowaną analizę kategorii...")

for message in consumer:
    tx = message.value
    category = tx.get('category', 'N/A')
    amount = tx.get('amount', 0.0)

    # Aktualizacja statystyk dla kategorii
    stats = category_stats[category]
    stats['count'] += 1
    stats['total'] += amount
    stats['min'] = min(stats['min'], amount)
    stats['max'] = max(stats['max'], amount)

    msg_count += 1

    # Co 10 wiadomości generujemy czytelny raport
    if msg_count % 10 == 0:
        print(f"\n--- STATYSTYKI PER KATEGORIA (Po {msg_count} msg) ---")
        header = f"{'Kategoria':<15} | {'Liczba':<7} | {'Suma':<10} | {'Min':<8} | {'Max':<8}"
        print(header)
        print("-" * len(header))
        
        for cat, data in sorted(category_stats.items()):
            print(f"{cat:<15} | {data['count']:<7} | {data['total']:<10.2f} | {data['min']:<8.2f} | {data['max']:<8.2f}")

Writing consumer_stats.py
